In [53]:
import pandas as pd
import os
from pathlib import Path
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from difflib import SequenceMatcher
import pandas as pd
from datetime import datetime, timedelta
from difflib import SequenceMatcher
import matplotlib.pyplot as plt
import os
from pathlib import Path
import seaborn as sns
import numpy as np
from lifelines.utils import to_long_format
from lifelines.utils import add_covariate_to_timeline
from lifelines import CoxTimeVaryingFitter
from statsmodels.distributions.empirical_distribution import ECDF
from scipy.stats import ks_2samp
from scipy import stats
from matplotlib.ticker import MultipleLocator
import matplotlib.ticker as mtick
import warnings

In [54]:
def son_similares(cadena1, cadena2, umbral=0.6):
    similitud = SequenceMatcher(None, cadena1, cadena2).ratio()
    return similitud >= umbral

In [55]:
path_actual = Path.cwd()
path_data = path_actual.parent / 'Data'

In [56]:
trib_publi = pd.read_excel(path_data/"EstadoCargaIEEH_SS 2024_16092024.xlsx")
trib_priv = pd.read_excel(path_data/"EstadoCargaIEEH_SEREMI 2024_16092024.xlsx")#, skiprows=2

In [ ]:
trib = pd.concat([trib_publi,trib_priv])
trib['Servicio de Salud'] = trib['Servicio de Salud'].fillna(method='ffill')
trib['SEREMI de Salud'] = trib['SEREMI de Salud'].fillna(method='ffill')
df_hospitales = trib.dropna(subset=['Nombre Establecimiento'])

regiones_seremi = {
    'SEREMI de Arica y Parinacota': 'ARICA Y PARINACOTA',
    'SEREMI de Tarapacá': 'TARAPACA',
    'SEREMI de Antofagasta': 'ANTOFAGASTA',
    'SEREMI de Atacama': 'ATACAMA',
    'SEREMI de Coquimbo': 'COQUIMBO',
    'SEREMI de Valparaíso': 'VALPARAISO',
    'SEREMI Metropolitana de Santiago': 'METROPOLITANA',
    'SEREMI del Libertador B. O Higgins': "O'HIGGINS",
    'SEREMI del Maule': 'MAULE',
    'SEREMI de Ñuble': 'NUBLE',
    'SEREMI del Biobío': 'BIOBIO',
    'SEREMI de La Araucanía': 'ARAUCANIA',
    'SEREMI de Los Rios': 'LOS RIOS',
    'SEREMI de Los Lagos': 'LOS LAGOS',
    'SEREMI de Magallanes y de la Antártica Chilena': 'MAGALLANES Y ANTARTICA'
    }

regiones_servicio= {
    'Arica y Parinacota': 'ARICA Y PARINACOTA',
    'Tarapacá': 'TARAPACA',
    'Antofagasta': 'ANTOFAGASTA',
    'Atacama': 'ATACAMA',
    'Coquimbo': 'COQUIMBO',
    'Valparaíso San Antonio': 'VALPARAISO',
    'Viña del Mar Quillota': 'VALPARAISO',
    'Aconcagua': 'VALPARAISO',
    'Metropolitano Norte': 'METROPOLITANA',
    'Metropolitano Occidente': 'METROPOLITANA',
    'Metropolitano Central': 'METROPOLITANA',
    'Metropolitano Oriente': 'METROPOLITANA',
    'Metropolitano Sur': 'METROPOLITANA',
    'Metropolitano Sur Oriente': 'METROPOLITANA',
    'Del Libertador B.O Higgins': "O'HIGGINS",
    'Del Maule': 'MAULE',
    'Ñuble': 'NUBLE',
    'Concepción': 'BIOBIO',
    'Arauco': 'BIOBIO',
    'Talcahuano': 'BIOBIO',
    'Biobío': 'BIOBIO',
    'Araucanía Norte': 'ARAUCANIA',
    'Araucanía Sur': 'ARAUCANIA',
    'Los Ríos': 'LOS RIOS',
    'Osorno': 'LOS LAGOS',
    'Del Reloncaví': 'LOS LAGOS',
    'Chiloé': 'LOS LAGOS',
    'Aisén': 'AISEN',
    'Magallanes': 'MAGALLANES Y ANTARTICA',
    'Ignorado' : None,
    'Extranjero' : None,
    }

def mapear_regiont(region, regiones_dict, umbral=0.9):
    if pd.isna(region):
        return None  # Si la región es NaN o None, devuelve None
    for key, value in regiones_dict.items():
        if son_similares(region, key, umbral):
            return value
    return None  # Si no hay coincidencia, devolver None

df_hospitales['region_seremi'] = df_hospitales['SEREMI de Salud'].apply(lambda x: mapear_regiont(x, regiones_seremi))
df_hospitales['region_servicio'] = df_hospitales['Servicio de Salud'].apply(lambda x: mapear_regiont(x, regiones_servicio))

df_hospitales['region_final'] = df_hospitales['region_seremi'].combine_first(df_hospitales['region_servicio'])

df_hospitales = df_hospitales.drop(columns=['region_seremi', 'region_servicio'])

In [101]:
df_regional = df_hospitales[['region_final', 'Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep',
       'Oct', 'Nov', 'Dic']]
agrup_sum=df_regional.groupby(['region_final']).sum()
maximos_por_region_df = agrup_sum.max(axis=1).reset_index()
maximos_por_region_df.columns = ['region_final', 'maximo']
df_merged = agrup_sum.merge(maximos_por_region_df, on='region_final', how='left').set_index('region_final')
df_dividido = df_merged.div(df_merged['maximo'], axis=0).round(2)

In [60]:
regions_2023 = pd.read_csv(path_data/'completed_region_summaries.csv')
regions_2023 = regions_2023[['Region', 'Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep',
                             'Oct', 'Nov', 'Dec']].rename(columns={'Dec':'Dic'})
regions_2023['Region'] = regions_2023['Region'].apply(lambda x: mapear_regiont(x, regiones_servicio))
regions_2023 = regions_2023.groupby(['Region']).sum().reset_index().rename(columns={'Region':'region_final'}).set_index('region_final')

In [62]:
regions_2023_seremi = pd.read_csv(path_data/'seremi_region_summaries.csv')
regions_2023_seremi = regions_2023_seremi[['Region', 'Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep',
                             'Oct', 'Nov', 'Dic']]
regions_2023_seremi['Region'] = regions_2023_seremi['Region'].apply(lambda x: mapear_regiont(x, regiones_seremi))
regions_2023_seremi = regions_2023_seremi.groupby(['Region']).sum().reset_index().rename(columns={'Region':'region_final'}).set_index('region_final')


In [72]:
trib_2023 = pd.concat([regions_2023,regions_2023_seremi]).groupby('region_final').sum()

In [ ]:
df_comparacion = agrup_sum.copy()  # Crear una copia del DataFrame de 2024 para almacenar los resultados

# Iterar sobre cada columna de mes (Ene, Feb, ... Dic)
for mes in ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']:
    # Calcular el porcentaje de cambio respecto a 2023
    df_comparacion[mes] = (agrup_sum[mes] / trib_2023[mes])

df_exito=df_comparacion.copy()

for mes in ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul']:
    # Calcular el porcentaje de cambio respecto a 2023
    df_exito = df_exito[df_exito[mes] >=0.8]
df_exito

In [85]:
df_comparacion['Promedio_Mensual_hasta julio'] = df_comparacion[['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul']].min(axis=1)

In [ ]:
df_comparacion

In [ ]:
df_comparacion['Promedio_Mensual_hasta julio']

In [ ]:
agrup_sum[['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun','Jul']].std(axis=1)

In [104]:
meses=['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun','Jul','Ago']
#agrup_sum['tributa_bien'] = np.where(agrup_sum[meses].min(axis=1) >= agrup_sum[meses].mean(axis=1) - 2*agrup_sum[meses].std(axis=1),1,0)
agrup_sum['tributa_bien_jul'] = np.where(agrup_sum['Jul'] >= 0.95*agrup_sum['Jun'],1,0)
agrup_sum['tributa_bien_ago'] = np.where(agrup_sum['Ago'] >= 0.95*agrup_sum['Jul'],1,0)
agrup_sum


,Ene,Feb,Mar,Abr,May,Jun,Jul,Ago,Sep,Oct,Nov,Dic,tributa_bien_jul,tributa_bien_ago
region_final,,,,,,,,,,,,,,
AISEN,877.0,894.0,870.0,977.0,944.0,838.0,887.0,846.0,4.0,0.0,0.0,0.0,1,1
ANTOFAGASTA,4187.0,4024.0,4402.0,4361.0,4587.0,4268.0,4381.0,490.0,0.0,0.0,0.0,0.0,1,0
ARAUCANIA,7597.0,7119.0,7559.0,7799.0,8000.0,7419.0,6003.0,3340.0,0.0,0.0,0.0,0.0,0,0
ARICA Y PARINACOTA,1671.0,1581.0,1734.0,1681.0,1633.0,1519.0,1482.0,1514.0,206.0,0.0,0.0,0.0,1,1
ATACAMA,1715.0,1677.0,1787.0,1844.0,1869.0,1748.0,1850.0,1616.0,66.0,0.0,0.0,0.0,1,0
BIOBIO,14624.0,12613.0,13704.0,13744.0,13797.0,12760.0,12188.0,5454.0,132.0,0.0,0.0,0.0,1,0
COQUIMBO,4225.0,4339.0,4467.0,4477.0,4405.0,3997.0,3711.0,1653.0,0.0,0.0,0.0,0.0,0,0
LOS LAGOS,6690.0,6323.0,6479.0,6711.0,6120.0,5940.0,6244.0,2517.0,158.0,0.0,0.0,0.0,1,0
LOS RIOS,3298.0,3014.0,3241.0,3213.0,3279.0,3003.0,3178.0,528.0,146.0,0.0,0.0,0.0,1,0
